# VARSR — Inference Benchmark
This notebook:
1. Clones the code repo (always fresh, no zip uploads)
2. Installs dependencies
3. Downloads VARSR + VQVAE checkpoints from HuggingFace
4. Downloads RealSR test dataset (HR + LR pairs)
5. Runs **baseline** config vs **tuned** config
6. Prints a full metrics comparison table + saves visual comparison

**Each time a Kaggle session restarts:** just Run All — checkpoints are re-downloaded automatically (fast on Kaggle servers), code is always up to date via git clone.

## 0. Setup — clone code & verify GPU

In [ ]:
import os, torch

# ── CONFIGURE ─────────────────────────────────────────────────────
GITHUB_REPO = 'https://github.com/AhmedIzaan/VARSR.git'
WORK_DIR    = '/kaggle/working/VARSR'
HF_REPO     = 'qyp2000/VARSR'
# ──────────────────────────────────────────────────────────────────

# Clone / update repo
if os.path.exists(WORK_DIR):
    print('Repo exists — pulling latest changes...')
    os.system(f'git -C {WORK_DIR} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {GITHUB_REPO} {WORK_DIR}')

os.chdir(WORK_DIR)
print(f'\nWorking dir: {os.getcwd()}')
print('Files:', sorted(os.listdir('.')))

# Verify GPU
assert torch.cuda.is_available(), (
    'No GPU detected. Enable it: Notebook settings → Accelerator → GPU T4 x2'
)
import psutil
print(f'\nGPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'RAM  : {psutil.virtual_memory().total/1e9:.1f} GB')

## 1. Install dependencies

In [ ]:
%%bash
# NOTE: Do NOT pin numpy here — Kaggle's torchvision is compiled against numpy 2.x
# Downgrading numpy causes binary incompatibility errors
pip install -q \
    typed-argument-parser \
    pyiqa \
    pytz \
    'diffusers==0.32.2' \
    'accelerate==1.0.1' \
    'transformers==4.37.2' \
    safetensors \
    wandb \
    huggingface_hub \
    scikit-image \
    einops \
    lmdb \
    2>&1 | tail -3
echo "Dependencies installed."

## 2. Download checkpoints
Downloads VQVAE.pth (~1.4 GB) and VARSR.pth (~13.7 GB) from HuggingFace.
Kaggle's servers pull at ~100 MB/s so this takes ~2–3 minutes.

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('checkpoints', exist_ok=True)

for fname in ['VQVAE.pth', 'VARSR.pth']:
    dest = f'checkpoints/{fname}'
    if os.path.exists(dest):
        print(f'  already exists: {dest}  ({os.path.getsize(dest)/1e9:.2f} GB)')
        continue
    print(f'Downloading {fname}...')
    hf_hub_download(repo_id=HF_REPO, filename=fname, local_dir='checkpoints/')
    print(f'  -> {dest}  ({os.path.getsize(dest)/1e9:.2f} GB)')

print('\nCheckpoints ready.')

## 3. Prepare test dataset
Using **RealSR V3** — real camera LR/HR pairs, same degradation type the model was trained on.

**Before running this cell:** In the Kaggle right panel → **Add Data** → search **`realsr-v3`** → add the dataset by `yashchoudhary`.  
It mounts instantly at `/kaggle/input/realsr-v3/` — no download needed.

In [ ]:
import os, glob, shutil

os.makedirs('testset/RealSR/LR', exist_ok=True)
os.makedirs('testset/RealSR/HR', exist_ok=True)

hr_existing = glob.glob('testset/RealSR/HR/*.png')
lr_existing = glob.glob('testset/RealSR/LR/*.png')

if hr_existing and lr_existing:
    print(f'Already set up — HR: {len(hr_existing)}  LR: {len(lr_existing)}')
else:
    kaggle_root = '/kaggle/input/datasets/yashchoudhary/realsr-v3'

    if not os.path.exists(kaggle_root):
        raise FileNotFoundError(
            f'Dataset not found at {kaggle_root}\n'
            f'Available: {os.listdir("/kaggle/input")}'
        )

    print(f'Found dataset at {kaggle_root}')

    # RealSR V3 structure: RealSR(V3)/{Canon,Nikon}/Test/4/*_HR.png and *_LR4.png
    hr_src = sorted(glob.glob(f'{kaggle_root}/RealSR(V3)/*/Test/4/*_HR.png'))
    lr_src = sorted(glob.glob(f'{kaggle_root}/RealSR(V3)/*/Test/4/*_LR4.png'))

    print(f'Found {len(hr_src)} HR  and  {len(lr_src)} LR images')

    if not hr_src or not lr_src:
        raise FileNotFoundError(
            'Could not find HR or LR images. '
            'Check the glob patterns match the folder structure.'
        )

    if len(hr_src) != len(lr_src):
        print(f'WARNING: HR count ({len(hr_src)}) != LR count ({len(lr_src)}) — some pairs may be missing')

    for p in hr_src:
        shutil.copy(p, f'testset/RealSR/HR/{os.path.basename(p)}')
    for p in lr_src:
        shutil.copy(p, f'testset/RealSR/LR/{os.path.basename(p)}')

    print(f'Sample HR: {[os.path.basename(p) for p in hr_src[:3]]}')
    print(f'Sample LR: {[os.path.basename(p) for p in lr_src[:3]]}')

# Final check
hr_n = len(glob.glob('testset/RealSR/HR/*.png'))
lr_n = len(glob.glob('testset/RealSR/LR/*.png'))
assert hr_n > 0 and lr_n > 0, f'Dataset empty! HR={hr_n} LR={lr_n}'
print(f'\nDataset ready — HR: {hr_n}  LR: {lr_n}')

# Verify pairs match
print('\nPair verification:')
for hr, lr in zip(sorted(glob.glob('testset/RealSR/HR/*.png'))[:3],
                  sorted(glob.glob('testset/RealSR/LR/*.png'))[:3]):
    hr_id = os.path.basename(hr).replace('_HR.png', '')
    lr_id = os.path.basename(lr).replace('_LR4.png', '')
    match = '✓' if hr_id == lr_id else '✗ MISMATCH'
    print(f'  {match}  {os.path.basename(hr)}  →  {os.path.basename(lr)}')

## 4. Initialize distributed env (single GPU)

In [ ]:
import os, sys

os.environ['MASTER_ADDR'] = '127.0.0.1'
os.environ['MASTER_PORT'] = '29500'
os.environ['RANK']        = '0'
os.environ['WORLD_SIZE']  = '1'
os.environ['LOCAL_RANK']  = '0'

sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

import dist
from utils import arg_util
from test_varsr import main, metrics
print('Imports OK')

## 5. Run Baseline (original paper settings)

In [ ]:
BASELINE_TAG = 'VARPrediction_BASELINE'

# Check if already done (resume-safe)
import glob
existing = glob.glob(f'testset/*/{BASELINE_TAG}/*.png')
if existing:
    print(f'Baseline predictions already exist ({len(existing)} images) — skipping inference.')
    print('Computing metrics only...')
else:
    sys.argv = [
        'test_varsr.py',
        '--vae_model_path', 'checkpoints/VQVAE.pth',
        '--var_test_path',  'checkpoints/VARSR.pth',
        '--cfg',       '6.0',
        '--top_k',     '1',
        '--top_p',     '0.75',
        '--diff_temp', '1.0',
        '--color_fix', 'adain',
    ]
    args = arg_util.init_dist_and_get_args()
    print('Running baseline inference...')
    main(args, output_tag=BASELINE_TAG)
    print('Done.')

baseline_results = metrics(output_tag=BASELINE_TAG)
print('Baseline metrics computed.')

## 6. Run Tuned Config

In [ ]:
import importlib, test_varsr
importlib.reload(test_varsr)
from test_varsr import main, metrics

TUNED_TAG = 'VARPrediction_TUNED'

existing = glob.glob(f'testset/*/{TUNED_TAG}/*.png')
if existing:
    print(f'Tuned predictions already exist ({len(existing)} images) — skipping inference.')
    print('Computing metrics only...')
else:
    sys.argv = [
        'test_varsr.py',
        '--vae_model_path', 'checkpoints/VQVAE.pth',
        '--var_test_path',  'checkpoints/VARSR.pth',
        '--cfg',        '7.0',
        '--top_k',      '900',
        '--top_p',      '0.95',
        '--diff_temp',  '0.7',
        '--color_fix',  'wavelet',
        '--infer_seed', '42',
    ]
    args = arg_util.init_dist_and_get_args()
    print('Running tuned inference...')
    main(args, output_tag=TUNED_TAG)
    print('Done.')

tuned_results = metrics(output_tag=TUNED_TAG)
print('Tuned metrics computed.')

## 7. Comparison Table

In [ ]:
import pandas as pd

def avg_results(d):
    if not d: return {}
    keys = list(next(iter(d.values())).keys())
    return {k: sum(r[k] for r in d.values()) / len(d) for k in keys}

b = avg_results(baseline_results)
t = avg_results(tuned_results)

HIGHER = {'psnr', 'ssim', 'clip_iqa', 'musiq', 'maniqa'}
LOWER  = {'lpips', 'dists', 'niqe', 'fid'}

rows = []
for m in ['psnr', 'ssim', 'lpips', 'dists', 'niqe', 'clip_iqa', 'musiq', 'maniqa', 'fid']:
    bv, tv = b.get(m, float('nan')), t.get(m, float('nan'))
    diff   = tv - bv
    better = diff > 0 if m in HIGHER else diff < 0
    rows.append({
        'Metric':    m.upper(),
        'Goal':      '↑ higher' if m in HIGHER else '↓ lower',
        'Baseline':  round(bv, 4),
        'Tuned':     round(tv, 4),
        'Δ':         f"{'+' if diff > 0 else ''}{diff:.4f}",
        '':          '✅' if better else ('➖' if abs(diff) < 1e-4 else '❌'),
    })

df = pd.DataFrame(rows)
print('\n' + '='*60)
print('  VARSR  |  Baseline vs Tuned  (avg across test folders)')
print('='*60)
print(df.to_string(index=False))
print('='*60)

df.to_csv('comparison_results.csv', index=False)
print('\nSaved → comparison_results.csv')

## 8. Visual Comparison (LR / Baseline / Tuned)

In [ ]:
import glob, random
from PIL import Image
import matplotlib.pyplot as plt

lr_paths = sorted(glob.glob('testset/*/LR/*.png') + glob.glob('testset/*/LR/*.jpg'))
samples  = random.sample(lr_paths, min(4, len(lr_paths)))

fig, axes = plt.subplots(len(samples), 3, figsize=(18, 6 * len(samples)))
if len(samples) == 1: axes = [axes]

for row, lr_path in enumerate(samples):
    root, fname = os.path.split(lr_path)
    root = root.replace('/LR', '')
    imgs = [
        (Image.open(lr_path).convert('RGB'),
         'LR Input'),
        (Image.open(f'{root}/{BASELINE_TAG}/{fname}').convert('RGB') if os.path.exists(f'{root}/{BASELINE_TAG}/{fname}') else None,
         'Baseline\ncfg=6.0 | adain | temp=1.0'),
        (Image.open(f'{root}/{TUNED_TAG}/{fname}').convert('RGB')    if os.path.exists(f'{root}/{TUNED_TAG}/{fname}')    else None,
         'Tuned\ncfg=7.0 | wavelet | temp=0.7'),
    ]
    for col, (img, title) in enumerate(imgs):
        if img:
            axes[row][col].imshow(img)
        else:
            axes[row][col].text(0.5, 0.5, 'missing', ha='center', va='center')
        axes[row][col].set_title(title, fontsize=12, fontweight='bold')
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('visual_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → visual_comparison.png')

## 9. Download results

In [ ]:
from IPython.display import FileLink, display
for f in ['comparison_results.csv', 'visual_comparison.png']:
    if os.path.exists(f):
        print(f'Download: ', end='')
        display(FileLink(f))
    else:
        print(f'Not found: {f}')